# 第 5 讲练习：队列、`deque` 与 BFS

先说明队列中每个元素代表什么、何时入队、何时标记访问，再开始编码。

## 1. 概念题

1. 为什么 BFS 通常使用 `deque`，而不使用 `list.pop(0)`？
2. 为什么通常应该在入队时标记 `visited`？
3. 普通 BFS 什么时候能保证最短距离？
4. 按层处理时，为什么要在循环开始时保存 `level_size`？
5. 图有 `V` 个节点和 `E` 条边时，邻接表 BFS 的时间复杂度是什么？

在这里作答。

答：1.因为list.pop(0)操作复杂度是O(n)
2.这样避免后边再次搜到这个节点，导致进入queue多次   
3.无权图   
4.保证这一层只搜这么多的节点，剩下的都是下一个level的节点了   
5.O(V+E)  

## 2. RecentCounter：维护时间窗口

每次调用 `ping(t)`，返回区间 `[t-3000, t]` 内的请求数量。调用中的 `t` 严格递增。

使用 `deque`，每个请求最多入队和出队一次。

维护一个queue，每次ping就记录到que中，然后检查有没有t-3000之前的数据，输出掉。最后返回que的长度

In [1]:
from collections import deque

class RecentCounter:
    def __init__(self):
        self.data = deque()

    def ping(self, t):
        self.data.append(t)
        while self.data[0] < t-3000:
            self.data.popleft()
        return len(self.data)

In [2]:
counter = RecentCounter()
assert counter.ping(1) == 1
assert counter.ping(100) == 2
assert counter.ping(3001) == 3
assert counter.ping(3002) == 3

## 3. 无权图最短距离

图由邻接表给出。返回 `start` 到所有可达节点的最短边数，结果为字典。

提示：距离字典可以同时充当 `visited`。

graph是一个字典。   
从start开始，作为que的第一个元素，建立distance字典   
对于que中的每个元素，依次popleft，找邻居  
对于没有距离记录每个邻居，存放进dis字典中，距离就是这个元素的dis+1   
最后返回distance

In [3]:
from collections import deque
def shortest_distances(graph, start):
    que = deque([start])
    dis = {start:0}

    while que:
        x = que.popleft()
        for y in graph[x]:
            if y not in dis:
                dis[y] = dis[x]+1
                que.append(y)
    return dis

In [4]:
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D'],
    'C': ['A', 'D'],
    'D': ['B', 'C', 'E'],
    'E': ['D'],
    'F': [],
}
assert shortest_distances(graph, 'A') == {'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 3}
assert shortest_distances(graph, 'F') == {'F': 0}

## 4. 网格最短路

网格中 `0` 可以通行，`1` 是障碍。只能上下左右移动。返回从 `start` 到 `target` 的最少步数，不可达返回 `-1`。

坐标使用 `(row, col)` 元组。不要修改输入网格。

que的初始状态是[start，0]，增加一个int dis记录当前步数，维护一个visit set记录来过的点   
对于que中每个对象依次检查邻居，对邻居的要求：1.没在visit中。2.没有出界。3.数值不是1   
然后记录邻居进visit，进que


In [13]:
from collections import deque

def grid_shortest_path(grid, start, target):
    row = start[0]
    col = start[1]
    que = deque([[row,col,0]])
    visit = {tuple(start)}

    maxr = len(grid)
    maxc = len(grid[0])
    dir = [
        [0,1],
        [0,-1],
        [1,0],
        [-1,0]
    ]

    if tuple(start) == tuple(target):
        return 0
    while que:
        node = que.popleft()
        dis  = node[2]

        for x in dir:
            nr = node[0]+x[0]
            nc = node[1]+x[1]

            if tuple([nr, nc]) == tuple(target):
                return dis + 1

            if -1 < nr < maxr and -1 < nc < maxc and tuple([nr, nc]) not in visit and grid[nr][nc] == 0:
                visit.add(tuple([nr, nc]))
                que.append([nr, nc, dis + 1])

    return -1

In [14]:
grid = [
    [0, 0, 1, 0],
    [1, 0, 0, 0],
    [0, 0, 1, 0],
]
assert grid_shortest_path(grid, (0, 0), (0, 0)) == 0
assert grid_shortest_path(grid, (0, 0), (2, 3)) == 5
assert grid_shortest_path([[0, 1], [1, 0]], (0, 0), (1, 1)) == -1

## 5. 挑战：腐烂的橘子

`0` 是空格，`1` 是新鲜橘子，`2` 是腐烂橘子。每分钟腐烂橘子会感染上下左右的新鲜橘子。返回全部腐烂所需最少分钟；无法全部腐烂返回 `-1`。

这是多源 BFS：所有初始腐烂橘子应同时入队。允许修改输入网格。

第一步，统计start的烂橘子，初始化que，统计fresh数量   
开始传播，对于que中的每个元素，都寻找上下左右的邻居，邻居要求： 1.没在visit中。2.没有出界。  
如果邻居数值是1，改成2，fresh-1  
fresh == 0时，传播完毕，返回时间    


In [21]:
def oranges_rotting(grid):
    fresh = 0
    que = deque()
    visit = set()
    for row in range(len(grid)):
        for col in range(len(grid[0])):
            if grid[row][col] == 1:
                fresh+=1
            if grid[row][col] == 2:
                que.append((row,col,0))
                visit.add((row,col))

    if fresh == 0: return 0
    dir = [
        [0,1],
        [0,-1],
        [1,0],
        [-1,0]
    ]
    
    maxr = len(grid)
    maxc = len(grid[0])
    
    while que:
        node = que.popleft()
        row = node[0]
        col = node[1]
        dis = node[2]
        
        for x in dir:
            nr = node[0]+x[0]
            nc = node[1]+x[1]

            if -1 < nr < maxr and -1 < nc < maxc and (nr, nc) not in visit and grid[nr][nc] != 0:
                if grid[nr][nc] == 1:
                    fresh-=1
                    if fresh == 0: return dis+1
                
                visit.add((nr, nc))
                que.append((nr, nc, dis + 1))
    return -1
                

优化：判断语句这个grid[nr][nc] != 0可以改成grid[nr][nc] = 1，然后删除visit这set，直接使用grid中的数字来标记是否经过，已经经过的路径都变成2

In [22]:
assert oranges_rotting([[2,1,1],[1,1,0],[0,1,1]]) == 4
assert oranges_rotting([[2,1,1],[0,1,1],[1,0,1]]) == -1
assert oranges_rotting([[0,2]]) == 0
assert oranges_rotting([[1]]) == -1

# 累计复习

## 6. 哈希复习：第一个只出现一次的字符

返回字符串中第一个只出现一次的字符下标；不存在返回 `-1`。要求 `O(n)` 时间。

一定需要遍历完才知道，用一个dict储存每个字母和对应的下标
O(n)并非只能遍历一次，只要不是嵌套循环，遍历多次也可以。


In [23]:
def first_unique_char(text):
    count = {}
    for ch in text:
        if ch in count: count[ch] +=1
        else:count[ch] = 1
    for i in range(len(text)):
        if count[text[i]] == 1:
            return i
    return -1

In [24]:
assert first_unique_char('leetcode') == 0
assert first_unique_char('loveleetcode') == 2
assert first_unique_char('aabb') == -1
assert first_unique_char('') == -1

## 7. 栈复习：简化退格字符串

字符 `#` 表示删除此前一个字符；如果没有字符则什么也不做。返回处理后的字符串。

示例：`ab#c → ac`，`a##c → c`。

遍历text，碰到字母，就加入栈，碰到#，如果栈中有元素就pop，最后返回栈



In [25]:
def apply_backspaces(text):
    stack = []
    
    for ch in text:
        if ch != "#":
            stack.append(ch)
        elif stack:
            stack.pop()
    
    return "".join(stack)

In [26]:
assert apply_backspaces('ab#c') == 'ac'
assert apply_backspaces('a##c') == 'c'
assert apply_backspaces('####') == ''
assert apply_backspaces('abc') == 'abc'

## 8. 总结

1. 哪道题需要多源 BFS？
2. 哪些结构同时承担了 `visited` 的作用？
3. 网格 BFS 的节点和边分别是什么？
4. 每个状态最多入队几次？
5. 复习题中你是否能不看旧讲义完成哈希和栈代码？

答：1.腐烂橘子这种多起点题目   
2.记录dis的字典   
3.节点是每个格子，边是direction这个移动的向量   
4.最多入队一次   
5.这两个复习题都比较简单，可以独立做出来